---
## Statistical Analysis 
---

### About this notebook 
This notebook applies formal hypothesis tests to the four most
business-critical patterns identified in Notebook 01 and reports
findings with test statistics, p-values, effect sizes, and plain
English business interpretations.

**Statistical reporting standard for every test:**
1. Test statistic - the numerical output of the test
2. P-value - probability this result occurred by chance
3. Effect size - how large the difference actually is
4. Plain English - what it means for the business

**Why all four, not just p-value:**
A p-value tells you whether a difference is real.
An effect size tells you whether it matters.
A finding that is statistically significant with a negligible
effect size is real but practically useless to a business.
Both numbers together tell the complete story.

**Hypotheses tested in this notebook:**
1. Late delivery is associated with significantly lower review scores
2. High-value RFM customers behave statistically differently
3. Review scores vary significantly by product category
4. Seasonal revenue patterns are statistically significant

In [3]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import warnings 
import os

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize']    = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titlesize']    = 13
plt.rcParams['axes.titleweight']  = 'bold'

COLOUR_PRIMARY   = '#1B3A5C'
COLOUR_ACCENT    = '#2E75B6'
COLOUR_HIGHLIGHT = '#E74C3C'
COLOUR_NEUTRAL   = '#95A5A6'
COLOUR_GREEN     = '#2ECC71'
COLOUR_EMPHASIS  = '#d00000'

os.makedirs('../outputs/figures', exist_ok=True)




In [4]:
#loading the data 

master = pd.read_csv(
    '../data/processed/master.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
        'order_approved_at'
    ]
)

rfm_base = pd.read_csv(
    '../data/processed/rfm_base.csv',
    parse_dates=['first_order', 'last_order']
)

In [5]:
print(f"master:   {master.shape[0]:,} rows x {master.shape[1]} columns")
print(f"rfm_base: {rfm_base.shape[0]:,} rows x {rfm_base.shape[1]} columns")
print("\nReady.")

master:   113,425 rows x 29 columns
rfm_base: 96,136 rows x 10 columns

Ready.


---
## Hypothesis One : Late delivery produces lower review scores 
---

**Null hypothesis (H₀):**
There is no difference in review scores between orders that
arrived late and orders that arrived on time or early.
 
**Alternative hypothesis (H₁):**
Orders that arrived late have significantly lower review scores
than orders that arrived on time or early.

### The Decision Rule
If p-value < 0.05: reject H₀ — the difference is real.
If p-value ≥ 0.05: fail to reject H₀ — cannot confirm the difference.
"""

In [6]:
master['order_status'].head(15)

0     delivered
1     delivered
2     delivered
3     delivered
4     delivered
5     delivered
6      invoiced
7     delivered
8     delivered
9     delivered
10    delivered
11    delivered
12    delivered
13    delivered
14    delivered
Name: order_status, dtype: str

In [10]:
delivered = master[master['order_status'] == 'delivered'].copy()

delivered_orders = delivered.drop_duplicates(subset='order_id').copy()

reviewed_delivered = delivered_orders.dropna(subset=['review_score']).copy()

In [25]:
on_time_scores = reviewed_delivered[
    reviewed_delivered['was_late'] == False
]['review_score']

late_scores = reviewed_delivered[
    reviewed_delivered['was_late'] == True
]['review_score']



In [24]:
print("Group sizes:")
print(f"  On-time / early orders: {len(on_time_scores):,}")
print(f"  Late orders:            {len(late_scores):,}")
print(f"  Total reviewed:         {len(reviewed_delivered):,}")
 
print("\nGroup statistics:")
print(f"  On-time mean score: {on_time_scores.mean():.2f}")
print(f"  Late mean score:    {late_scores.mean():.2f}")
print(f"  Raw difference:     {on_time_scores.mean() - late_scores.mean():.2f} points")
print(f"  On-time median:     {on_time_scores.median():.1f}")
print(f"  Late median:        {late_scores.median():.1f}")

Group sizes:
  On-time / early orders: 89,451
  Late orders:            89,451
  Total reviewed:         95,832

Group statistics:
  On-time mean score: 4.29
  Late mean score:    4.29
  Raw difference:     0.00 points
  On-time median:     5.0
  Late median:        5.0


In [26]:
print("\nOn-time score distribution (%):")
print((on_time_scores.value_counts(normalize=True).sort_index() * 100).round(1))
print("\nLate score distribution (%):")
print((late_scores.value_counts(normalize=True).sort_index() * 100).round(1))


On-time score distribution (%):
review_score
1.0     6.6
2.0     2.6
3.0     8.1
4.0    20.4
5.0    62.3
Name: proportion, dtype: float64

Late score distribution (%):
review_score
1.0    53.8
2.0     8.6
3.0    10.9
4.0    10.2
5.0    16.5
Name: proportion, dtype: float64


In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)
 
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])
 
score_colours = {
    1: '#E74C3C',
    2: '#E67E22',
    3: '#95A5A6',
    4: '#2ECC71',
    5: '#1B3A5C'
}
 
# ── Panel 1: On-time distribution ─────────────────────────
on_time_pct = on_time_scores.value_counts(normalize=True).sort_index() * 100
 
bars1 = ax1.bar(
    on_time_pct.index,
    on_time_pct.values,
    color=[score_colours[s] for s in on_time_pct.index],
    edgecolor='white',
    width=0.6
)
for bar, val in zip(bars1, on_time_pct.values):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f'{val:.1f}%',
        ha='center', va='bottom',
        fontsize=9, fontweight='bold'
    )
ax1.set_title(
    f'On-Time / Early Orders\n(n={len(on_time_scores):,}  |  mean={on_time_scores.mean():.2f})',
    fontweight='bold'
)
ax1.set_xlabel('Review Score')
ax1.set_ylabel('% of Orders')
ax1.set_xticks([1,2,3,4,5])
ax1.set_xticklabels(['1\nVery Bad','2\nBad','3\nNeutral','4\nGood','5\nExcellent'], fontweight='semibold')
ax1.set_ylim(0, on_time_pct.max() * 1.15)
 
# ── Panel 2: Late distribution ─────────────────────────────
late_pct = late_scores.value_counts(normalize=True).sort_index() * 100
 
bars2 = ax2.bar(
    late_pct.index,
    late_pct.values,
    color=[score_colours[s] for s in late_pct.index],
    edgecolor='white',
    width=0.6
)
for bar, val in zip(bars2, late_pct.values):
    ax2.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f'{val:.1f}%',
        ha='center', va='bottom',
        fontsize=9, fontweight='bold'
    )
ax2.set_title(
    f'Late Orders\n(n={len(late_scores):,}  |  mean={late_scores.mean():.2f})',
    fontweight='bold'
)
ax2.set_xlabel('Review Score')
ax2.set_ylabel('% of Orders')
ax2.set_xticks([1,2,3,4,5])
ax2.set_xticklabels(['1\nVery Bad','2\nBad','3\nNeutral','4\nGood','5\nExcellent'], fontweight='semibold')
ax2.set_ylim(0, late_pct.max() * 1.15)
all_scores = pd.DataFrame({
    'On-Time / Early': on_time_pct,
    'Late':            late_pct
}).fillna(0)
 
x     = np.arange(1, 6)
width = 0.35

# ── Panel 3: Review Score Distribution: On-Time vs Late — Side by Side ─────────────────────────────
bars3a = ax3.bar(
    x - width/2,
    all_scores['On-Time / Early'],
    width=width,
    color=COLOUR_PRIMARY,
    alpha=0.85,
    label=f'On-Time / Early (mean={on_time_scores.mean():.2f})'
)
bars3b = ax3.bar(
    x + width/2,
    all_scores['Late'],
    width=width,
    color=COLOUR_EMPHASIS,
    alpha=0.85,
    label=f'Late (mean={late_scores.mean():.2f})'
)
 
ax3.set_xlabel('Review Score')
ax3.set_ylabel('% of Orders in Group')
ax3.set_title(
    'Review Score Distribution: On-Time vs Late — Side by Side',
    fontweight='bold'
)
ax3.set_xticks(x)
ax3.set_xticklabels(['1 — Very Bad','2 — Bad','3 — Neutral','4 — Good','5 — Excellent'], fontweight='semibold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3, linestyle='--')
 
plt.suptitle(
    'Hypothesis 1 — Review Score Distributions\nOn-Time vs Late Delivery',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(
    '../outputs/figures/h1_score_distributions.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print("Chart saved.")